In [1]:
# ==============================
# Imports
# ==============================
%pip install openpyxl

from __future__ import annotations

import json
import math
import os
import random
import sys
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import numpy as np
import pandas as pd
import codecarbon
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights, ResNet50_Weights

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    auc as sk_auc,
    silhouette_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from ocpc_py import MultiClassPC

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==============================
# Configuração global
# ==============================

SEED = 42
TASK = "Ischaemia"
DATA_ROOT = Path("../data/ischaemia")
MODEL_NAME = "efficientnet"   # "efficientnet" ou "resnet50"
IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
N_FOLDS = 5
VAL_N_SPLITS = 4  # usado para criar a validação dentro do conjunto de treino+val
PCA_N_COMPONENTS = 50
NUM_WORKERS = 0
SAVE_GRADCAM_EXAMPLES = True
MAX_GRADCAM_IMAGES_PER_FOLD = 10
USE_CODECARBON = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT_OUTPUT = Path("../reports")
DIR_MODELS = Path("../models") / TASK.lower()
DIR_METRICS = ROOT_OUTPUT / "metrics"
DIR_FIGURES = ROOT_OUTPUT / "figures"
DIR_PCA_FIG = DIR_FIGURES / "pca"
DIR_GRADCAM = DIR_FIGURES / "gradcam"
DIR_EMISSIONS = ROOT_OUTPUT / "emissions"

for directory in [DIR_MODELS, DIR_METRICS, DIR_FIGURES, DIR_PCA_FIG, DIR_GRADCAM, DIR_EMISSIONS]:
    directory.mkdir(parents=True, exist_ok=True)

FOLD_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
CLASS_COLORS = {0: "#3B8BD4", 1: "#E8593C"}
SPLIT_COLORS = {"train": "#4C72B0", "val": "#55A868", "test": "#DD8452"}

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# ==============================
# Dataset e transformações
# ==============================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN.tolist(), std=IMAGENET_STD.tolist()),
])


class DFUDataset(Dataset):

    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths: List[Path] = []
        self.labels: List[int] = []
        self.identifiers: List[str] = []

        positive_dir = self.root_dir / "Aug-Positive"
        negative_dir = self.root_dir / "Aug-Negative"

        if not positive_dir.exists() or not negative_dir.exists():
            raise FileNotFoundError(
                f"Estrutura de pastas não encontrada em {self.root_dir}. "
                "Esperado: Aug-Positive e Aug-Negative."
            )

        for class_dir, label in [(positive_dir, 1), (negative_dir, 0)]:
            for image_path in sorted(class_dir.iterdir()):
                if image_path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                    continue
                patient_id = image_path.stem.split("_")[0]
                self.image_paths.append(image_path)
                self.labels.append(label)
                self.identifiers.append(patient_id)

        if len(self.image_paths) == 0:
            raise RuntimeError(f"Nenhuma imagem encontrada em {self.root_dir}")

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int):
        image_path = self.image_paths[index]
        image = Image.open(image_path).convert("RGB")
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        # Mantemos o caminho para facilitar Grad-CAM e auditoria do fold.
        return image, label, str(image_path)

# ==============================
# Utilidades de split
# ==============================

def build_subset(dataset: DFUDataset, indices: Sequence[int], transform) -> Dataset:
    """
    Cria um subset com transform específico.

    Em vez de mudar o transform do dataset original no meio do pipeline,
    clonamos uma visão do dataset com o transform apropriado.
    """
    subset_dataset = DFUDataset(dataset.root_dir, transform=transform)
    return Subset(subset_dataset, list(indices))


def get_nested_group_split(
    labels: Sequence[int],
    groups: Sequence[str],
    n_outer_splits: int = N_FOLDS,
    n_inner_splits: int = VAL_N_SPLITS,
    seed: int = SEED,
):
    """
    Gera splits aninhados:
    - outer: train_val vs test
    - inner: train vs val, apenas dentro do conjunto train_val

    Ambos respeitam estratificação aproximada por rótulo e separação por grupos.
    """
    labels = np.asarray(labels)
    groups = np.asarray(groups)

    outer_cv = StratifiedGroupKFold(n_splits=n_outer_splits, shuffle=True, random_state=seed)

    for outer_fold, (train_val_idx, test_idx) in enumerate(
        outer_cv.split(np.zeros(len(labels)), labels, groups), start=1
    ):
        inner_labels = labels[train_val_idx]
        inner_groups = groups[train_val_idx]

        inner_cv = StratifiedGroupKFold(n_splits=n_inner_splits, shuffle=True, random_state=seed + outer_fold)
        inner_train_rel, inner_val_rel = next(
            inner_cv.split(np.zeros(len(train_val_idx)), inner_labels, inner_groups)
        )

        train_idx = np.array(train_val_idx)[inner_train_rel]
        val_idx = np.array(train_val_idx)[inner_val_rel]

        yield outer_fold, train_idx, val_idx, np.array(test_idx)

In [4]:
# ==============================
# Modelo
# ==============================

def create_model(model_name: str = MODEL_NAME) -> nn.Module:
    """
    Cria o backbone de transfer learning.

    A extração da penúltima camada será feita via hook no avgpool:
    - EfficientNet-B0 -> 1280 features
    - ResNet-50       -> 2048 features
    """
    model_name = model_name.lower()

    if model_name == "resnet50":
        model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        num_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_features, 1),
        )
    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_features, 1),
        )
    else:
        raise ValueError(f"MODEL_NAME inválido: {model_name}")

    return model.to(DEVICE)


def get_penultimate_hook_layer(model: nn.Module) -> nn.Module:
    """
    Retorna a camada onde o hook de features deve ser registrado.

    Usamos a saída do avgpool, que é justamente o embedding global imediatamente
    antes da cabeça classificadora.
    """
    if hasattr(model, "avgpool"):
        return model.avgpool

    raise AttributeError("Não foi possível localizar a camada de avgpool do modelo.")


def get_gradcam_target_layer(model: nn.Module) -> nn.Module:
    """
    Escolhe uma camada convolucional profunda para Grad-CAM.
    """
    model_name = MODEL_NAME.lower()

    if model_name == "resnet50":
        return model.layer4[-1]

    if model_name in {"efficientnet", "efficientnet_b0"}:
        return model.features[-1]

    raise ValueError(f"Grad-CAM não implementado para {MODEL_NAME}")

In [5]:

# ==============================
# Métricas
# ==============================

def safe_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, y_score))
    except Exception:
        return float("nan")


def safe_pr_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    try:
        return float(average_precision_score(y_true, y_score))
    except Exception:
        return float("nan")


def calculate_binary_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> Dict[str, object]:
    """
    Calcula métricas clássicas de classificação binária.
    """
    y_true = np.asarray(y_true).astype(int).ravel()
    y_score = np.asarray(y_score).astype(float).ravel()
    y_pred = (y_score >= threshold).astype(int)

    return {
        "auc": safe_roc_auc(y_true, y_score),
        "pr_auc": safe_pr_auc(y_true, y_score),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "n_samples": int(len(y_true)),
        "positive_rate": float(np.mean(y_true)),
    }


def format_metrics_for_log(metrics: Dict[str, object], prefix: str = "") -> str:
    """
    Formata as métricas principais em uma única string para os logs.
    """
    lead = f"{prefix} " if prefix else ""
    return (
        f"{lead}acc={metrics['accuracy']:.4f} | "
        f"precision={metrics['precision']:.4f} | "
        f"recall={metrics['recall']:.4f} | "
        f"f1={metrics['f1']:.4f} | "
        f"auc={metrics['auc']:.4f}"
    )


def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, Dict[str, object], np.ndarray, np.ndarray]:
    """
    Avalia a CNN em um DataLoader e retorna:
    - loss média
    - dicionário de métricas
    - y_true
    - y_score
    """
    model.eval()
    losses: List[float] = []
    all_scores: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []

    with torch.no_grad():
        for inputs, labels, _paths in loader:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            logits = model(inputs)
            loss = criterion(logits, labels)

            scores = torch.sigmoid(logits).cpu().numpy().ravel()
            y_true = labels.cpu().numpy().ravel()

            losses.append(loss.item())
            all_scores.append(scores)
            all_labels.append(y_true)

    y_true_all = np.concatenate(all_labels) if all_labels else np.array([], dtype=int)
    y_score_all = np.concatenate(all_scores) if all_scores else np.array([], dtype=float)
    metrics = calculate_binary_metrics(y_true_all, y_score_all)

    return float(np.mean(losses) if losses else float("nan")), metrics, y_true_all, y_score_all


In [6]:

# ==============================
# Treinamento
# ==============================

@dataclass
class TrainingHistory:
    train_loss: List[float] = field(default_factory=list)
    val_loss: List[float] = field(default_factory=list)
    val_auc: List[float] = field(default_factory=list)
    val_f1: List[float] = field(default_factory=list)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    fold: int,
) -> Tuple[TrainingHistory, Path]:
    """
    Treina a CNN com early stopping baseado em val_loss.

    Salvamos o checkpoint selecionado do fold, que será usado depois na avaliação
    e na extração de features.
    """
    history = TrainingHistory()
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    checkpoint_path = DIR_MODELS / f"{TASK}_{MODEL_NAME}_fold{fold}_selected.pth"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        n_seen = 0

        for inputs, labels, _paths in train_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            n_seen += batch_size

        train_loss = running_loss / max(n_seen, 1)

        val_loss, val_metrics, _, _ = evaluate_model(model, val_loader, criterion)

        history.train_loss.append(float(train_loss))
        history.val_loss.append(float(val_loss))
        history.val_auc.append(float(val_metrics["auc"]))
        history.val_f1.append(float(val_metrics["f1"]))

        print(
            f"Fold {fold:02d} | Epoch {epoch:03d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_precision={val_metrics['precision']:.4f} | "
            f"val_recall={val_metrics['recall']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping no fold {fold}: {PATIENCE} épocas sem melhora.")
            break

    return history, checkpoint_path


In [7]:
# ==============================
# Extração de features e PCA
# ==============================

def extract_features(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Extrai o embedding da penúltima camada usando forward hook.

    Retorna:
    - X_features: ndarray (N, D)
    - y_labels: ndarray (N,)
    - paths: lista de caminhos das imagens
    """
    hook_layer = get_penultimate_hook_layer(model)
    feature_batches: List[torch.Tensor] = []
    label_batches: List[np.ndarray] = []
    path_list: List[str] = []

    def hook_fn(_module, _inputs, output):
        feature_batches.append(output.flatten(start_dim=1).detach().cpu())

    handle = hook_layer.register_forward_hook(hook_fn)

    model.eval()
    with torch.no_grad():
        for inputs, labels, paths in loader:
            inputs = inputs.to(DEVICE)
            _ = model(inputs)
            label_batches.append(np.asarray(labels))
            path_list.extend(list(paths))

    handle.remove()

    X_features = torch.cat(feature_batches, dim=0).numpy() if feature_batches else np.empty((0, 0))
    y_labels = np.concatenate(label_batches) if label_batches else np.array([], dtype=int)

    return X_features, y_labels, path_list


def fit_pca_on_train(
    X_train: np.ndarray,
    n_components: int = PCA_N_COMPONENTS,
) -> Tuple[StandardScaler, PCA, np.ndarray, np.ndarray, np.ndarray]:
    """
    Ajusta scaler + PCA apenas no treino.
    """
    effective_components = min(n_components, X_train.shape[0], X_train.shape[1])
    if effective_components < 2:
        raise ValueError(
            "O número efetivo de componentes PCA ficou < 2. "
            "Verifique o tamanho do conjunto de treino e a dimensionalidade das features."
        )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    pca = PCA(n_components=effective_components, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train_scaled)

    explained = pca.explained_variance_ratio_
    cumulative = np.cumsum(explained)
    return scaler, pca, X_train_pca, explained, cumulative


def apply_pca(
    scaler: StandardScaler,
    pca: PCA,
    X: np.ndarray,
) -> np.ndarray:
    """
    Transforma um novo conjunto usando scaler e PCA já ajustados no treino.
    """
    X_scaled = scaler.transform(X)
    return pca.transform(X_scaled)

In [8]:

# ==============================
# OCPC (MultiClassPC com patch de compatibilidade)
# ==============================

def normalize_binary_probabilities(prob_matrix: np.ndarray) -> np.ndarray:
    """
    Extrai a coluna da classe positiva para o caso binário.
    """
    prob_matrix = np.asarray(prob_matrix, dtype=float)
    if prob_matrix.ndim == 1:
        return prob_matrix
    if prob_matrix.shape[1] == 1:
        return prob_matrix[:, 0]
    return prob_matrix[:, 1]


def _patch_ocpc_map_to_arcl() -> None:
    """
    Corrige incompatibilidades do ocpc_py com versões recentes do NumPy.

    O pacote original cria `lengths` como vetor coluna (shape=(k,1)) e depois
    tenta somar `lengths[idx]` diretamente a um escalar. Em NumPy recente isso
    pode gerar:
        TypeError: only 0-dimensional arrays can be converted to Python scalars

    Este patch:
    1) força `lengths` a ser 1D;
    2) aceita tanto a chamada map_to_arcl(curve, X) quanto
       map_to_arcl(edges, vertices, X), pois o código do pacote usa os dois padrões.
    """
    try:
        import ocpc_py.utils as ocpc_utils
        import ocpc_py.Models as ocpc_models
        from copy import copy as shallow_copy
    except Exception:
        return

    def map_to_arcl_compat(*args):
        if len(args) == 2:
            curve, x = args
            edges = shallow_copy(curve.edges)
            vertices = shallow_copy(curve.vertices)
            close = getattr(curve, "close", False)
            start_curve = getattr(curve, "start_curve", None)
            end_curve = getattr(curve, "end_curve", None)
        elif len(args) == 3:
            edges, vertices, x = args
            edges = shallow_copy(edges)
            vertices = shallow_copy(vertices)
            close = False
            start_curve = None
            end_curve = None
        else:
            raise TypeError("map_to_arcl_compat espera 2 ou 3 argumentos.")

        x = np.ascontiguousarray(np.asarray(x, dtype=np.float64))
        vertices = np.ascontiguousarray(np.asarray(vertices, dtype=np.float64))
        edges = np.asarray(edges)

        if x.ndim != 2:
            raise ValueError(f"X deve ser 2D em map_to_arcl_compat, recebido {x.shape}.")
        if vertices.ndim != 2:
            raise ValueError(f"vertices deve ser 2D, recebido {vertices.shape}.")
        if edges.ndim != 2:
            raise ValueError(f"edges deve ser 2D, recebido {edges.shape}.")

        n = x.shape[0]
        d_features = x.shape[1]
        n_segments = max(int(edges.shape[0] - 1), 0)

        if n_segments == 0:
            y = np.zeros((n, d_features + 1), dtype=np.float64)
            d = np.zeros(n, dtype=np.float64)
            return y, d

        segments = np.zeros((d_features, 2, n_segments), dtype=np.float64)
        e = edges.copy()
        lengths = np.zeros(n_segments + 1, dtype=np.float64)

        if close is False:
            degree_two_vertices = np.where(np.sum(e, axis=0) == 2)[0]
            if degree_two_vertices.size == 0:
                i = 0
            else:
                i = int(degree_two_vertices[0])
            next_vertices = np.where(e[i, :] > 0)[0]
            if next_vertices.size == 0:
                j = 0
            else:
                j = int(next_vertices[0])
        else:
            i = int(start_curve if start_curve is not None else 0)
            j = int(end_curve if end_curve is not None else 0)

        segment = 0
        while segment < n_segments:
            e[i, j] = 0
            e[j, i] = 0

            a = np.reshape(vertices[:, i], (d_features, 1))
            b = np.reshape(vertices[:, j], (d_features, 1))
            segments[:, :, segment] = np.concatenate((a, b), axis=1)

            lengths[segment + 1] = lengths[segment] + float(np.linalg.norm(vertices[:, i] - vertices[:, j]))

            segment += 1
            i = j
            next_vertices = np.where(e[i, :] > 0)[0]
            if segment < n_segments:
                if next_vertices.size == 0:
                    break
                j = int(next_vertices[0])

        y = np.zeros((n, d_features + 1), dtype=np.float64)
        dists = np.zeros((n, n_segments), dtype=np.float64)
        rest = np.zeros((n, d_features + 1, n_segments), dtype=np.float64)

        for seg_idx in range(n_segments):
            d, t, p = ocpc_utils.seg_dist(segments[:, 0, seg_idx], segments[:, 1, seg_idx], x.T)
            dists[:, seg_idx] = np.asarray(d, dtype=np.float64).reshape(-1)
            t = np.asarray(t, dtype=np.float64).reshape(-1, 1)
            p = np.asarray(p, dtype=np.float64)
            if p.ndim == 1:
                p = p.reshape(-1, 1)
            rest[:, :, seg_idx] = np.concatenate((p, t), axis=1)

        d = np.min(dists, axis=1)
        vr = np.argmin(dists, axis=1)

        for row_idx in range(n):
            seg_idx = int(vr[row_idx])
            y[row_idx, :] = rest[row_idx, :, seg_idx]
            y[row_idx, 0] = y[row_idx, 0] + lengths[seg_idx]

        return y, d

    ocpc_utils.map_to_arcl = map_to_arcl_compat
    ocpc_models.map_to_arcl = map_to_arcl_compat


def prepare_ocpc_inputs(
    X: np.ndarray,
    y: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Garante o formato exato esperado pelo MultiClassPC.
    """
    X = np.ascontiguousarray(np.asarray(X, dtype=np.float64))
    if X.ndim != 2:
        raise ValueError(f"X deve ser 2D no OCPC. Shape recebido: {X.shape}")

    if y is None:
        return X, None

    y = np.asarray(y).reshape(-1)
    if y.ndim != 1:
        raise ValueError(f"y deve ser 1D no OCPC. Shape recebido: {y.shape}")
    if X.shape[0] != y.shape[0]:
        raise ValueError(f"Número de amostras inconsistente: X={X.shape[0]} e y={y.shape[0]}")
    if np.unique(y).size < 2:
        raise ValueError("O MultiClassPC requer pelo menos duas classes no treino.")

    # Mantemos classes como inteiros 0/1, exatamente como o MultiClassPC aceita.
    y = np.ascontiguousarray(y.astype(np.int32))
    return X, y


def fit_ocpc_classifier(
    X_train_pca: np.ndarray,
    y_train: np.ndarray,
) -> object:
    """
    Ajusta exatamente o MultiClassPC do pacote ocpc_py.
    """
    _patch_ocpc_map_to_arcl()

    X_train_pca, y_train = prepare_ocpc_inputs(X_train_pca, y_train)

    clf = MultiClassPC()
    clf.fit(X_train_pca, y_train)
    return clf


def predict_with_ocpc(
    clf: object,
    X_eval_pca: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Gera predições e scores do MultiClassPC treinado.
    """
    X_eval_pca, _ = prepare_ocpc_inputs(X_eval_pca, None)

    y_pred = np.asarray(clf.predict(X_eval_pca)).astype(int).ravel()

    if hasattr(clf, "predict_proba"):
        try:
            y_score = normalize_binary_probabilities(clf.predict_proba(X_eval_pca))
        except Exception:
            y_score = y_pred.astype(float)
    else:
        y_score = y_pred.astype(float)

    y_score = np.nan_to_num(np.asarray(y_score, dtype=float), nan=0.0, posinf=1.0, neginf=0.0)
    return y_score, y_pred


In [ ]:

# =============================================================================
# CÉLULA 8 — CAMs qualitativos para o paper
# Salva arquivos separados por método:
#   - GradCAM
#   - GradCAM++
#   - EigenCAM
# Sem máscaras, sem métricas de sobreposição e sem sanity checks.
# =============================================================================

from pathlib import Path
from typing import Dict, Literal

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    GradCAM = None
    GradCAMPlusPlus = None
    EigenCAM = None
    show_cam_on_image = None


CAM_METHODS = {
    "GradCAM": GradCAM,
    "GradCAMPlusPlus": GradCAMPlusPlus,
    "EigenCAM": EigenCAM,
}


def get_gradcam_target_layer(model: nn.Module) -> nn.Module:
    """
    Escolhe a camada convolucional-alvo para os métodos CAM.
    """
    model_name = MODEL_NAME.lower()

    if model_name == "resnet50":
        return model.layer4[-1].conv3

    if model_name in {"efficientnet", "efficientnet_b0"}:
        return model.features[-1]

    raise ValueError(f"CAM não implementado para {MODEL_NAME}")


class BinaryClassifierTarget:
    """
    Target para classificador binário com 1 logit.

    target_class = 1 -> explica evidência para classe positiva
    target_class = 0 -> explica evidência para classe negativa usando -logit
    """
    def __init__(self, target_class: int):
        if target_class not in (0, 1):
            raise ValueError("target_class deve ser 0 ou 1.")
        self.target_class = target_class

    def __call__(self, model_output: torch.Tensor) -> torch.Tensor:
        if model_output.ndim == 2:
            logit = model_output[:, 0]
        elif model_output.ndim == 1:
            logit = model_output
        else:
            raise ValueError(f"Formato inesperado de model_output: {model_output.shape}")

        return logit if self.target_class == 1 else -logit


def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    """Converte tensor normalizado (C,H,W) para RGB float32 em [0,1]."""
    image = image_tensor.detach().cpu().float().numpy().transpose(1, 2, 0)
    image = (image * IMAGENET_STD) + IMAGENET_MEAN
    image = np.clip(image, 0.0, 1.0)
    return image.astype(np.float32)


def normalize_cam(cam: np.ndarray) -> np.ndarray:
    """Normaliza mapa CAM para [0,1]."""
    cam = cam.astype(np.float32)
    cam_min = cam.min()
    cam_max = cam.max()

    if cam_max - cam_min < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)

    return (cam - cam_min) / (cam_max - cam_min)


def cam_to_colormap(cam: np.ndarray) -> np.ndarray:
    """Converte heatmap [0,1] para RGB colorido."""
    cam_uint8 = np.uint8(255 * normalize_cam(cam))
    heatmap_bgr = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)
    return heatmap_rgb


def predict_binary(model: nn.Module, image_tensor: torch.Tensor):
    """
    Faz predição binária para uma única imagem (C,H,W).

    Retorna:
    - logit
    - probabilidade da classe 1
    - probabilidade da classe 0
    - classe prevista
    - confiança da classe prevista
    """
    model.eval()
    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(input_tensor)

        if logits.ndim == 2:
            logit = logits[:, 0]
        else:
            logit = logits

        prob_pos = torch.sigmoid(logit)[0].item()
        prob_neg = 1.0 - prob_pos
        pred = int(prob_pos >= 0.5)
        pred_conf = prob_pos if pred == 1 else prob_neg

    return float(logit[0].item()), float(prob_pos), float(prob_neg), pred, float(pred_conf)


def get_confusion_group(true: int, pred: int) -> str:
    if true == 1 and pred == 1:
        return "TP"
    if true == 0 and pred == 0:
        return "TN"
    if true == 0 and pred == 1:
        return "FP"
    if true == 1 and pred == 0:
        return "FN"
    raise ValueError(f"Combinação inválida: true={true}, pred={pred}")


def generate_cam_outputs(
    model: nn.Module,
    image_tensor: torch.Tensor,
    target_class: int,
    cam_method_cls,
):
    """
    Gera heatmap e overlay para um método CAM.
    """
    if show_cam_on_image is None:
        raise ImportError("pytorch-grad-cam não está disponível no ambiente.")

    model.eval()
    target_layer = get_gradcam_target_layer(model)

    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad_(True)
    targets = [BinaryClassifierTarget(target_class=target_class)]

    with cam_method_cls(model=model, target_layers=[target_layer]) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    grayscale_cam = normalize_cam(grayscale_cam)
    rgb_image = denormalize_image(image_tensor)
    heatmap_rgb = cam_to_colormap(grayscale_cam)
    overlay = show_cam_on_image(rgb_image, grayscale_cam, use_rgb=True)

    return heatmap_rgb, overlay


def save_cam_figure(
    original_rgb: np.ndarray,
    heatmap_rgb: np.ndarray,
    overlay: np.ndarray,
    out_path: Path,
    method_name: str,
    group: str,
    true: int,
    pred: int,
    prob_pos: float,
    prob_neg: float,
    pred_conf: float,
    explained_label: str,
) -> None:
    """Salva figura com 3 colunas: original, heatmap e overlay."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(original_rgb)
    axes[0].set_title(
        f"Original
Method={method_name} | Group={group} | True={true} | Pred={pred} | "
        f"P1={prob_pos:.3f} | P0={prob_neg:.3f} | Conf={pred_conf:.3f}"
    )
    axes[0].axis("off")

    axes[1].imshow(heatmap_rgb)
    axes[1].set_title(f"Heatmap
{explained_label}")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def save_gradcam_examples_by_group(
    model: nn.Module,
    dataset: Dataset,
    fold: int,
    max_per_group: int = 3,
    explain_mode: Literal["pred", "true"] = "pred",
    groups_to_save: tuple = ("TP", "TN", "FP", "FN"),
) -> None:
    """
    Salva exemplos CAM separados por TP, TN, FP e FN.
    Para cada imagem selecionada, salva um arquivo separado para cada método.
    """
    if not SAVE_GRADCAM_EXAMPLES:
        return

    available_methods = {
        name: cls for name, cls in CAM_METHODS.items() if cls is not None
    }
    if len(available_methods) == 0:
        print("Nenhum método CAM disponível no ambiente; pulando visualização.")
        return

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model.eval()
    saved_count: Dict[str, int] = {g: 0 for g in ("TP", "TN", "FP", "FN")}

    for image, label, path in loader:
        image_cpu = image.squeeze(0).cpu()
        true = int(label.numpy().ravel()[0])

        _, prob_pos, prob_neg, pred, pred_conf = predict_binary(model, image_cpu)
        group = get_confusion_group(true, pred)

        if group not in groups_to_save:
            continue
        if saved_count[group] >= max_per_group:
            continue

        if explain_mode == "pred":
            target_class = pred
            explained_label = f"Explained=Pred({pred})"
        elif explain_mode == "true":
            target_class = true
            explained_label = f"Explained=True({true})"
        else:
            raise ValueError("explain_mode deve ser 'pred' ou 'true'.")

        original_rgb = denormalize_image(image_cpu)
        fname = Path(path[0]).stem

        saved_any = False
        for method_name, cam_method_cls in available_methods.items():
            try:
                heatmap_rgb, overlay = generate_cam_outputs(
                    model=model,
                    image_tensor=image_cpu,
                    target_class=target_class,
                    cam_method_cls=cam_method_cls,
                )
            except Exception as e:
                print(f"Erro ao gerar {method_name} para {path[0]}: {e}")
                continue

            out_path = DIR_GRADCAM / (
                f"fold{fold}_{group}_{fname}_{explain_mode}_{method_name}.png"
            )
            save_cam_figure(
                original_rgb=original_rgb,
                heatmap_rgb=heatmap_rgb,
                overlay=overlay,
                out_path=out_path,
                method_name=method_name,
                group=group,
                true=true,
                pred=pred,
                prob_pos=prob_pos,
                prob_neg=prob_neg,
                pred_conf=pred_conf,
                explained_label=explained_label,
            )
            saved_any = True

        if saved_any:
            saved_count[group] += 1

        if all(saved_count[g] >= max_per_group for g in groups_to_save):
            break

    print(f"
CAMs salvos para fold {fold}:")
    for g in ("TP", "TN", "FP", "FN"):
        if g in groups_to_save:
            print(f"  {g}: {saved_count[g]} imagem(ns)")


def save_gradcam_grid_by_group(
    model: nn.Module,
    dataset: Dataset,
    fold: int,
    max_per_group: int = 2,
    explain_mode: Literal["pred", "true"] = "pred",
    groups_to_save: tuple = ("TP", "TN", "FP", "FN"),
) -> None:
    """
    Salva uma grade resumida usando apenas GradCAM.
    Mantida por compatibilidade e inspeção rápida.
    """
    if not SAVE_GRADCAM_EXAMPLES or GradCAM is None:
        return

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    collected = {g: [] for g in ("TP", "TN", "FP", "FN")}

    for image, label, path in loader:
        image_cpu = image.squeeze(0).cpu()
        true = int(label.numpy().ravel()[0])
        _, prob_pos, prob_neg, pred, pred_conf = predict_binary(model, image_cpu)
        group = get_confusion_group(true, pred)

        if group not in groups_to_save:
            continue
        if len(collected[group]) >= max_per_group:
            continue

        target_class = pred if explain_mode == "pred" else true
        explained_label = f"Pred({pred})" if explain_mode == "pred" else f"True({true})"

        try:
            heatmap_rgb, overlay = generate_cam_outputs(
                model=model,
                image_tensor=image_cpu,
                target_class=target_class,
                cam_method_cls=GradCAM,
            )
        except Exception as e:
            print(f"Erro ao gerar grade GradCAM para {path[0]}: {e}")
            continue

        collected[group].append({
            "original": denormalize_image(image_cpu),
            "heatmap": heatmap_rgb,
            "overlay": overlay,
            "true": true,
            "pred": pred,
            "prob_pos": prob_pos,
            "prob_neg": prob_neg,
            "pred_conf": pred_conf,
            "explained": explained_label,
            "name": Path(path[0]).stem,
        })

        if all(len(collected[g]) >= max_per_group for g in groups_to_save):
            break

    total_rows = sum(len(collected[g]) for g in groups_to_save)
    if total_rows == 0:
        print("Nenhuma imagem encontrada para montar a grade.")
        return

    fig, axes = plt.subplots(total_rows, 3, figsize=(12, 4 * total_rows))
    if total_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    row = 0
    for group in groups_to_save:
        for item in collected[group]:
            axes[row, 0].imshow(item["original"])
            axes[row, 0].set_title(
                f"{group} | Original
True={item['true']} | Pred={item['pred']} | "
                f"P1={item['prob_pos']:.3f} | P0={item['prob_neg']:.3f} | Conf={item['pred_conf']:.3f}"
            )
            axes[row, 0].axis("off")

            axes[row, 1].imshow(item["heatmap"])
            axes[row, 1].set_title(f"{group} | Heatmap
Explained={item['explained']}")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(item["overlay"])
            axes[row, 2].set_title(f"{group} | Overlay
{item['name']}")
            axes[row, 2].axis("off")
            row += 1

    plt.tight_layout()
    grid_path = DIR_GRADCAM / f"fold{fold}_grid_{explain_mode}_gradcam.png"
    plt.savefig(grid_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Grade GradCAM salva em: {grid_path}")


def save_gradcam_examples(
    model: nn.Module,
    dataset: Dataset,
    fold: int,
    max_images: int = 8,
) -> None:
    """
    Wrapper de compatibilidade com o pipeline antigo.
    """
    max_per_group = max(1, max_images // 4)

    save_gradcam_examples_by_group(
        model=model,
        dataset=dataset,
        fold=fold,
        max_per_group=max_per_group,
        explain_mode="pred",
        groups_to_save=("TP", "TN", "FP", "FN"),
    )

    save_gradcam_grid_by_group(
        model=model,
        dataset=dataset,
        fold=fold,
        max_per_group=min(2, max_per_group),
        explain_mode="pred",
        groups_to_save=("TP", "TN", "FP", "FN"),
    )


In [10]:
# ==============================
# Relatórios e visualizações
# ==============================

@dataclass
class FoldArtifact:
    fold: int
    train_history: TrainingHistory
    train_metrics_cnn: Dict[str, object]
    val_metrics_cnn: Dict[str, object]
    test_metrics_cnn: Dict[str, object]
    val_metrics_ocpc: Dict[str, object]
    test_metrics_ocpc: Dict[str, object]
    geometric_metrics_test: Dict[str, float]
    explained_variance_ratio: np.ndarray
    cumulative_variance: np.ndarray
    pca_train_2d: np.ndarray
    pca_val_2d: np.ndarray
    pca_test_2d: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    ocpc_test_scores: np.ndarray
    cnn_test_scores: np.ndarray
    emissions_kg: float


class ReportManager:
    """
    Organiza resultados por fold e gera tabelas/figuras consolidadas.

    As visualizações foram pensadas para ajudar tanto na análise do experimento
    quanto na redação do paper.
    """

    def __init__(self):
        self.artifacts: List[FoldArtifact] = []

    def add(self, artifact: FoldArtifact) -> None:
        self.artifacts.append(artifact)

    def finalize(self) -> None:
        self._export_metrics_csv()
        self._plot_training_history()
        self._plot_metric_panels()
        self._plot_confusion_matrices()
        self._plot_roc_pr_curves()
        self._plot_pca_scatter_by_fold()
        self._plot_pca_variance()
        self._plot_ocpc_score_distributions()
        self._plot_emissions()

    def _export_metrics_csv(self) -> None:
        rows = []
        for art in self.artifacts:
            row = {"fold": art.fold, "emissions_kg_co2eq": art.emissions_kg}

            for prefix, metrics in [
                ("cnn_train", art.train_metrics_cnn),
                ("cnn_val", art.val_metrics_cnn),
                ("cnn_test", art.test_metrics_cnn),
                ("ocpc_val", art.val_metrics_ocpc),
                ("ocpc_test", art.test_metrics_ocpc),
                ("geo_test", art.geometric_metrics_test),
            ]:
                for key, value in metrics.items():
                    if key == "confusion_matrix":
                        continue
                    row[f"{prefix}_{key}"] = value

            rows.append(row)

        df = pd.DataFrame(rows)
        summary = {"fold": "mean±std"}
        for col in df.columns:
            if col == "fold":
                continue
            if pd.api.types.is_numeric_dtype(df[col]):
                summary[col] = f"{df[col].mean():.4f} ± {df[col].std():.4f}"

        out_df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
        out_df.to_csv(DIR_METRICS / f"consolidated_{TASK}_{MODEL_NAME}.csv", index=False)

        df.to_excel(DIR_METRICS / f"consolidated_{TASK}_{MODEL_NAME}.xlsx", index=False)

    def _plot_training_history(self) -> None:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        fig.suptitle(f"Histórico de treinamento por fold — {TASK} ({MODEL_NAME})", fontsize=13)

        for art in self.artifacts:
            color = FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)]
            epochs = np.arange(1, len(art.train_history.train_loss) + 1)

            axes[0].plot(epochs, art.train_history.train_loss, "--", color=color, alpha=0.8, label=f"Fold {art.fold} train")
            axes[0].plot(epochs, art.train_history.val_loss, "-", color=color, linewidth=2, label=f"Fold {art.fold} val")
            axes[1].plot(epochs, art.train_history.val_auc, color=color, linewidth=2, label=f"Fold {art.fold}")
            axes[2].plot(epochs, art.train_history.val_f1, color=color, linewidth=2, label=f"Fold {art.fold}")

        titles = ["Loss", "AUC-ROC (val)", "F1-score (val)"]
        ylabels = ["Loss", "AUC", "F1"]
        for ax, title, ylabel in zip(axes, titles, ylabels):
            ax.set_title(title)
            ax.set_xlabel("Época")
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.3)
            ax.legend(fontsize=7, ncol=2)

        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"training_history_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_metric_panels(self) -> None:
        metrics_to_show = ["auc", "pr_auc", "f1", "accuracy", "precision", "recall"]
        pretty_names = {
            "auc": "AUC-ROC",
            "pr_auc": "PR-AUC",
            "f1": "F1-score",
            "accuracy": "Accuracy",
            "precision": "Precision",
            "recall": "Recall",
        }

        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            fig, axes = plt.subplots(2, 3, figsize=(15, 8))
            fig.suptitle(f"{title_prefix} — métricas por fold ({TASK}, {MODEL_NAME})", fontsize=13)

            folds = [art.fold for art in self.artifacts]
            x = np.arange(len(folds))

            for ax, metric_name in zip(axes.ravel(), metrics_to_show):
                val_values = [getattr(art, f"val_metrics_{family}")[metric_name] for art in self.artifacts]
                test_values = [getattr(art, f"test_metrics_{family}")[metric_name] for art in self.artifacts]

                bars1 = ax.bar(x - 0.18, val_values, width=0.36, label="Val", color="#4C72B0", alpha=0.8)
                bars2 = ax.bar(x + 0.18, test_values, width=0.36, label="Test", color="#DD8452", alpha=0.8)

                ax.set_xticks(x)
                ax.set_xticklabels([f"F{f}" for f in folds])
                ax.set_ylim(0.0, 1.08)
                ax.set_title(pretty_names[metric_name])
                ax.grid(axis="y", alpha=0.3)
                ax.legend(fontsize=8)

                for bar in list(bars1) + list(bars2):
                    value = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.3f}",
                            ha="center", va="bottom", fontsize=7)

            plt.tight_layout()
            plt.savefig(DIR_FIGURES / f"{family}_metric_panel_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            plt.close(fig)

    def _plot_confusion_matrices(self) -> None:
        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            for split in ["val", "test"]:
                n = len(self.artifacts)
                fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
                if n == 1:
                    axes = [axes]

                fig.suptitle(f"{title_prefix} — confusion matrices ({split})", fontsize=12)

                for ax, art in zip(axes, self.artifacts):
                    cm = getattr(art, f"{split}_metrics_{family}")["confusion_matrix"].astype(float)
                    row_sums = cm.sum(axis=1, keepdims=True)
                    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)

                    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
                    for r in range(cm.shape[0]):
                        for c in range(cm.shape[1]):
                            color = "white" if cm_norm[r, c] > 0.55 else "black"
                            ax.text(c, r, f"{int(cm[r, c])}\n({cm_norm[r, c]:.0%})",
                                    ha="center", va="center", color=color, fontsize=9)

                    ax.set_title(f"Fold {art.fold}")
                    ax.set_xticks([0, 1])
                    ax.set_yticks([0, 1])
                    ax.set_xticklabels(["Pred 0", "Pred 1"])
                    ax.set_yticklabels(["True 0", "True 1"])

                plt.tight_layout()
                plt.savefig(DIR_FIGURES / f"{family}_{split}_confusion_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
                plt.close(fig)

    def _plot_roc_pr_curves(self) -> None:
        """
        Curvas ROC/PR do conjunto de teste por fold.

        Essas figuras costumam ser muito úteis em artigos quando a base é
        potencialmente desbalanceada.
        """
        for family, title_prefix in [("cnn", "CNN"), ("ocpc", "OCPC")]:
            fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
            fig_pr, ax_pr = plt.subplots(figsize=(6, 5))

            for art in self.artifacts:
                y_true = art.y_test
                y_score = art.cnn_test_scores if family == "cnn" else art.ocpc_test_scores

                if len(np.unique(y_true)) < 2:
                    continue

                fpr, tpr, _ = roc_curve(y_true, y_score)
                precision, recall, _ = precision_recall_curve(y_true, y_score)

                roc_auc = safe_roc_auc(y_true, y_score)
                pr_auc = safe_pr_auc(y_true, y_score)
                color = FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)]

                ax_roc.plot(fpr, tpr, color=color, linewidth=2, label=f"Fold {art.fold} (AUC={roc_auc:.3f})")
                ax_pr.plot(recall, precision, color=color, linewidth=2, label=f"Fold {art.fold} (AP={pr_auc:.3f})")

            ax_roc.plot([0, 1], [0, 1], "--", color="gray", alpha=0.7)
            ax_roc.set_title(f"{title_prefix} — ROC no teste")
            ax_roc.set_xlabel("FPR")
            ax_roc.set_ylabel("TPR")
            ax_roc.grid(alpha=0.3)
            ax_roc.legend(fontsize=8)

            ax_pr.set_title(f"{title_prefix} — Precision-Recall no teste")
            ax_pr.set_xlabel("Recall")
            ax_pr.set_ylabel("Precision")
            ax_pr.grid(alpha=0.3)
            ax_pr.legend(fontsize=8)

            fig_roc.tight_layout()
            fig_pr.tight_layout()

            fig_roc.savefig(DIR_FIGURES / f"{family}_roc_test_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            fig_pr.savefig(DIR_FIGURES / f"{family}_pr_test_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
            plt.close(fig_roc)
            plt.close(fig_pr)

    def _plot_pca_scatter_by_fold(self) -> None:
        """
        Scatter de PC1 vs PC2 por fold, destacando train/val/test.

        Essa figura é útil no paper para mostrar:
        - distribuição das classes no espaço latente
        - diferença visual entre treino, validação e teste
        - coerência do embedding aprendido
        """
        n = len(self.artifacts)
        cols = min(3, n)
        rows = int(math.ceil(n / cols))

        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
        axes = np.array(axes).reshape(-1)

        for ax, art in zip(axes, self.artifacts):
            self._scatter_split(ax, art.pca_train_2d, art.y_train, "train")
            self._scatter_split(ax, art.pca_val_2d, art.y_val, "val")
            self._scatter_split(ax, art.pca_test_2d, art.y_test, "test")

            ax.set_title(f"Fold {art.fold}")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.grid(alpha=0.2)

        for ax in axes[len(self.artifacts):]:
            ax.set_visible(False)

        handles = [
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["train"], label="Train"),
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["val"], label="Val"),
            plt.Line2D([], [], linestyle="", marker="o", color=SPLIT_COLORS["test"], label="Test"),
        ]
        fig.legend(handles=handles, loc="upper center", ncol=3)
        fig.suptitle(f"PCA (PC1 × PC2) por fold — {TASK} ({MODEL_NAME})", fontsize=13)
        plt.tight_layout()
        plt.savefig(DIR_PCA_FIG / f"pca_split_scatter_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _scatter_split(self, ax, X_2d: np.ndarray, y: np.ndarray, split_name: str) -> None:
        color = SPLIT_COLORS[split_name]
        markers = {0: "o", 1: "^"}

        for cls in [0, 1]:
            mask = (y == cls)
            if np.sum(mask) == 0:
                continue
            ax.scatter(
                X_2d[mask, 0],
                X_2d[mask, 1],
                s=22,
                alpha=0.60 if split_name != "train" else 0.35,
                c=color,
                marker=markers[cls],
                edgecolors="none",
            )

    def _plot_pca_variance(self) -> None:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for art in self.artifacts:
            cumulative_pct = art.cumulative_variance * 100
            ax.plot(
                np.arange(1, len(cumulative_pct) + 1),
                cumulative_pct,
                linewidth=2,
                color=FOLD_COLORS[(art.fold - 1) % len(FOLD_COLORS)],
                label=f"Fold {art.fold}",
            )

        ax.axhline(90, linestyle="--", color="gray", alpha=0.7)
        ax.axhline(95, linestyle=":", color="gray", alpha=0.7)
        ax.set_xlabel("Número de componentes")
        ax.set_ylabel("Variância acumulada (%)")
        ax.set_title(f"Scree plot acumulado — {TASK} ({MODEL_NAME})")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(DIR_PCA_FIG / f"scree_cumulative_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_ocpc_score_distributions(self) -> None:
        """
        Boxplots dos scores do OCPC no teste, separados por classe real.
        """
        rows = []
        for art in self.artifacts:
            for score, label in zip(art.ocpc_test_scores, art.y_test):
                rows.append({
                    "fold": art.fold,
                    "score": float(score),
                    "label": int(label),
                })

        df = pd.DataFrame(rows)
        if df.empty:
            return

        fig, ax = plt.subplots(figsize=(9, 5))
        positions = []
        labels = []
        data = []

        current_pos = 1
        for fold in sorted(df["fold"].unique()):
            for cls in [0, 1]:
                subset = df[(df["fold"] == fold) & (df["label"] == cls)]["score"].values
                if len(subset) == 0:
                    continue
                positions.append(current_pos)
                labels.append(f"F{fold}\nC{cls}")
                data.append(subset)
                current_pos += 1
            current_pos += 0.5

        bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
        for patch, lab in zip(bp["boxes"], labels):
            cls = int(lab.split("\n")[1].replace("C", ""))
            patch.set_facecolor(CLASS_COLORS[cls])
            patch.set_alpha(0.6)

        ax.set_xticks(positions)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Score OCPC para classe positiva")
        ax.set_title(f"Distribuição dos scores do OCPC no teste — {TASK} ({MODEL_NAME})")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"ocpc_score_distribution_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

    def _plot_emissions(self) -> None:
        emissions = [art.emissions_kg for art in self.artifacts]
        if len(emissions) == 0:
            return

        folds = [art.fold for art in self.artifacts]
        cumulative = np.cumsum(emissions)
        total = float(np.sum(emissions))

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        axes[0].bar([f"Fold {f}" for f in folds], emissions, color=FOLD_COLORS[:len(folds)], alpha=0.85)
        axes[0].set_title("Emissões por fold")
        axes[0].set_ylabel("kg CO₂eq")
        axes[0].grid(axis="y", alpha=0.3)

        axes[1].plot(range(1, len(cumulative) + 1), cumulative, marker="o", linewidth=2)
        axes[1].fill_between(range(1, len(cumulative) + 1), cumulative, alpha=0.15)
        axes[1].set_xticks(range(1, len(cumulative) + 1))
        axes[1].set_xticklabels([f"Fold {f}" for f in folds])
        axes[1].set_title(f"Acumulado (total = {total * 1000:.1f} g CO₂eq)")
        axes[1].set_ylabel("kg CO₂eq")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(DIR_FIGURES / f"emissions_{TASK}_{MODEL_NAME}.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

        pd.DataFrame({"fold": folds, "kg_co2eq": emissions, "kg_co2eq_cumulative": cumulative}).to_csv(
            DIR_METRICS / f"emissions_{TASK}_{MODEL_NAME}.csv",
            index=False
        )

In [11]:
# Carbon tracking
# ==============================

def create_emissions_tracker(fold: int):
    """
    Cria o tracker de emissões de carbono.
    """
    return codecarbon.EmissionsTracker(
        output_dir=str(DIR_EMISSIONS),
        output_file=f"fold{fold:02d}_train_emissions.csv",
        log_level="error",
    )


In [12]:
# ==============================
# Métricas geométricas no espaço PCA
# ==============================

def compute_geometric_metrics(
    X_pca: np.ndarray,
    y_true: np.ndarray,
    y_score: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    """
    Calcula métricas geométricas simples no espaço PCA, úteis para análise do OCPC.

    Parâmetros
    ----------
    X_pca : np.ndarray
        Features já transformadas pelo PCA, shape (n_samples, n_components).
    y_true : np.ndarray
        Rótulos verdadeiros, shape (n_samples,).
    y_score : np.ndarray
        Scores/probabilidades da classe positiva, shape (n_samples,).
    y_pred : np.ndarray
        Predições binárias do modelo, shape (n_samples,).

    Retorna
    -------
    Dict[str, float]
        Dicionário com métricas geométricas e separabilidade.
    """
    X_pca = np.asarray(X_pca, dtype=np.float64)
    y_true = np.asarray(y_true).astype(int).ravel()
    y_score = np.asarray(y_score, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred).astype(int).ravel()

    metrics: Dict[str, float] = {
        "silhouette_all_pcs": np.nan,
        "silhouette_pc1_pc2": np.nan,
        "center_distance_pc1_pc2": np.nan,
        "intra_class_spread_pos_pc1_pc2": np.nan,
        "intra_class_spread_neg_pc1_pc2": np.nan,
        "score_mean_pos": np.nan,
        "score_mean_neg": np.nan,
        "score_separation": np.nan,
        "pred_match_rate": np.nan,
    }

    if X_pca.ndim != 2 or X_pca.shape[0] == 0:
        return metrics

    if len(y_true) != X_pca.shape[0]:
        raise ValueError("X_pca e y_true têm números diferentes de amostras.")

    if len(y_score) != X_pca.shape[0]:
        raise ValueError("X_pca e y_score têm números diferentes de amostras.")

    if len(y_pred) != X_pca.shape[0]:
        raise ValueError("X_pca e y_pred têm números diferentes de amostras.")

    unique_classes = np.unique(y_true)

    # Silhouette usando todos os PCs
    if len(unique_classes) > 1 and X_pca.shape[0] > len(unique_classes):
        try:
            metrics["silhouette_all_pcs"] = float(silhouette_score(X_pca, y_true))
        except Exception:
            pass

    # Análise geométrica em 2D (PC1 e PC2), se existirem
    if X_pca.shape[1] >= 2:
        X_2d = X_pca[:, :2]

        if len(unique_classes) > 1 and X_2d.shape[0] > len(unique_classes):
            try:
                metrics["silhouette_pc1_pc2"] = float(silhouette_score(X_2d, y_true))
            except Exception:
                pass

        pos_mask = y_true == 1
        neg_mask = y_true == 0

        if np.any(pos_mask) and np.any(neg_mask):
            pos_points = X_2d[pos_mask]
            neg_points = X_2d[neg_mask]

            pos_center = pos_points.mean(axis=0)
            neg_center = neg_points.mean(axis=0)

            metrics["center_distance_pc1_pc2"] = float(
                np.linalg.norm(pos_center - neg_center)
            )

            metrics["intra_class_spread_pos_pc1_pc2"] = float(
                np.mean(np.linalg.norm(pos_points - pos_center, axis=1))
            )
            metrics["intra_class_spread_neg_pc1_pc2"] = float(
                np.mean(np.linalg.norm(neg_points - neg_center, axis=1))
            )

    # Separação dos scores do modelo
    pos_scores = y_score[y_true == 1]
    neg_scores = y_score[y_true == 0]

    if len(pos_scores) > 0:
        metrics["score_mean_pos"] = float(np.mean(pos_scores))

    if len(neg_scores) > 0:
        metrics["score_mean_neg"] = float(np.mean(neg_scores))

    if len(pos_scores) > 0 and len(neg_scores) > 0:
        metrics["score_separation"] = float(np.mean(pos_scores) - np.mean(neg_scores))

    # Taxa de acerto entre predição e rótulo real
    metrics["pred_match_rate"] = float(np.mean(y_true == y_pred))

    return metrics

In [13]:

#==============================
# Pipeline principal
# ==============================

def run_cross_validation() -> None:
    set_seed(SEED)

    dataset_for_split = DFUDataset(DATA_ROOT, transform=eval_transform)
    labels = np.asarray(dataset_for_split.labels)
    groups = np.asarray(dataset_for_split.identifiers)

    report = ReportManager()
    split_rows = []

    for fold, train_idx, val_idx, test_idx in get_nested_group_split(labels, groups):
        print("\n" + "=" * 80)
        print(f"FOLD {fold}/{N_FOLDS}")
        print("=" * 80)

        # Registramos os caminhos de cada split para auditoria do paper.
        split_rows.extend([
            {"fold": fold, "split": "train", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in train_idx
        ])
        split_rows.extend([
            {"fold": fold, "split": "val", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in val_idx
        ])
        split_rows.extend([
            {"fold": fold, "split": "test", "path": str(dataset_for_split.image_paths[i]), "label": int(labels[i]), "group": groups[i]}
            for i in test_idx
        ])

        # Subsets com augmentation no treino e transformação determinística em val/test.
        train_dataset = build_subset(dataset_for_split, train_idx, train_transform)
        val_dataset = build_subset(dataset_for_split, val_idx, eval_transform)
        test_dataset = build_subset(dataset_for_split, test_idx, eval_transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        model = create_model(MODEL_NAME)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

        tracker = create_emissions_tracker(fold)
        tracker.start()
        history, checkpoint_path = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            fold=fold,
        )
        emissions_kg = float(tracker.stop() or 0.0)

        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=False))
        model.eval()

        # Avaliação da CNN
        train_eval_loader = DataLoader(
            build_subset(dataset_for_split, train_idx, eval_transform),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )

        _, train_metrics_cnn, y_train_eval, _y_train_score_cnn = evaluate_model(model, train_eval_loader, criterion)
        _, val_metrics_cnn, y_val, _y_val_score_cnn = evaluate_model(model, val_loader, criterion)
        _, test_metrics_cnn, y_test, y_test_score_cnn = evaluate_model(model, test_loader, criterion)

        print(f"[Fold {fold}] CNN TRAIN | {format_metrics_for_log(train_metrics_cnn)}")
        print(f"[Fold {fold}] CNN VAL   | {format_metrics_for_log(val_metrics_cnn)}")
        print(f"[Fold {fold}] CNN TEST  | {format_metrics_for_log(test_metrics_cnn)}")

        # Extração de features separada por split para evitar leakage.
        X_train_feat, y_train, _train_paths = extract_features(model, train_eval_loader)
        X_val_feat, y_val_feat, _val_paths = extract_features(model, val_loader)
        X_test_feat, y_test_feat, _test_paths = extract_features(model, test_loader)

        # Sanidade: labels da extração devem coincidir com os labels da avaliação.
        assert np.array_equal(np.asarray(y_train, dtype=int), y_train_eval.astype(int))
        assert np.array_equal(np.asarray(y_val_feat, dtype=int), y_val.astype(int))
        assert np.array_equal(np.asarray(y_test_feat, dtype=int), y_test.astype(int))

        scaler, pca, X_train_pca, explained, cumulative = fit_pca_on_train(X_train_feat, PCA_N_COMPONENTS)
        X_val_pca = apply_pca(scaler, pca, X_val_feat)
        X_test_pca = apply_pca(scaler, pca, X_test_feat)

        print(
            f"[Fold {fold}] PCA shapes | "
            f"train={X_train_pca.shape} | val={X_val_pca.shape} | test={X_test_pca.shape}"
        )

        # OCPC no espaço PCA: ajusta UMA vez no treino e reutiliza em val/test.
        ocpc_model = fit_ocpc_classifier(X_train_pca, y_train)
        y_val_score_ocpc, y_val_pred_ocpc = predict_with_ocpc(ocpc_model, X_val_pca)
        y_test_score_ocpc, y_test_pred_ocpc = predict_with_ocpc(ocpc_model, X_test_pca)

        val_metrics_ocpc = calculate_binary_metrics(y_val, y_val_score_ocpc)
        test_metrics_ocpc = calculate_binary_metrics(y_test, y_test_score_ocpc)
        geometric_metrics_test = compute_geometric_metrics(X_test_pca, y_test, y_test_score_ocpc, y_test_pred_ocpc)

        print(f"[Fold {fold}] OCPC VAL  | {format_metrics_for_log(val_metrics_ocpc)}")
        print(f"[Fold {fold}] OCPC TEST | {format_metrics_for_log(test_metrics_ocpc)}")

        # Salva features PCA do teste e do treino para análises externas.
        train_pca_df = pd.DataFrame(
            X_train_pca[:, :min(10, X_train_pca.shape[1])],
            columns=[f"PC{i+1}" for i in range(min(10, X_train_pca.shape[1]))],
        )
        train_pca_df["label"] = np.asarray(y_train, dtype=int)
        train_pca_df["split"] = "train"
        train_pca_df["fold"] = fold
        train_pca_df.to_csv(DIR_METRICS / f"pca_train_fold{fold}.csv", index=False)

        test_pca_df = pd.DataFrame(
            X_test_pca[:, :min(10, X_test_pca.shape[1])],
            columns=[f"PC{i+1}" for i in range(min(10, X_test_pca.shape[1]))],
        )
        test_pca_df["label"] = y_test.astype(int)
        test_pca_df["split"] = "test"
        test_pca_df["fold"] = fold
        test_pca_df.to_csv(DIR_METRICS / f"pca_test_fold{fold}.csv", index=False)

        # Grad-CAM opcional, útil para o paper.
        if SAVE_GRADCAM_EXAMPLES:
            try:
                save_gradcam_examples(model, test_dataset, fold=fold)
            except Exception as exc:
                print(f"Grad-CAM do fold {fold} não pôde ser salvo: {exc}")

        artifact = FoldArtifact(
            fold=fold,
            train_history=history,
            train_metrics_cnn=train_metrics_cnn,
            val_metrics_cnn=val_metrics_cnn,
            test_metrics_cnn=test_metrics_cnn,
            val_metrics_ocpc=val_metrics_ocpc,
            test_metrics_ocpc=test_metrics_ocpc,
            geometric_metrics_test=geometric_metrics_test,
            explained_variance_ratio=explained,
            cumulative_variance=cumulative,
            pca_train_2d=X_train_pca[:, :2],
            pca_val_2d=X_val_pca[:, :2],
            pca_test_2d=X_test_pca[:, :2],
            y_train=np.asarray(y_train, dtype=int),
            y_val=y_val.astype(int),
            y_test=y_test.astype(int),
            ocpc_test_scores=y_test_score_ocpc,
            cnn_test_scores=y_test_score_cnn,
            emissions_kg=emissions_kg,
        )
        report.add(artifact)

        print(
            f"[Fold {fold}] CNN TEST | {format_metrics_for_log(test_metrics_cnn)} | "
            f"OCPC TEST | {format_metrics_for_log(test_metrics_ocpc)} | "
            f"Silhouette={geometric_metrics_test['silhouette_pc1_pc2']:.4f} | "
            f"CO2={emissions_kg * 1000:.2f} g"
        )

    pd.DataFrame(split_rows).to_csv(DIR_METRICS / f"split_audit_{TASK}_{MODEL_NAME}.csv", index=False)
    report.finalize()
    print("Pipeline concluído com sucesso.")


In [ ]:

# ==============================
# CAMs — nota de consolidação
# ==============================
# Esta versão do notebook salva os mapas CAM em arquivos PNG separados por método.
# Não há etapa adicional de consolidação quantitativa nesta célula.


In [14]:
# Ponto de entrada
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_cross_validation()


FOLD 1/5


[codecarbon WARNING @ 20:41:20] Multiple instances of codecarbon are allowed to run at the same time.


Fold 01 | Epoch 001/30 | train_loss=0.2962 | val_loss=0.2049 | val_auc=0.9749 | val_f1=0.9177
Fold 01 | Epoch 002/30 | train_loss=0.0999 | val_loss=0.2022 | val_auc=0.9789 | val_f1=0.9322
Fold 01 | Epoch 003/30 | train_loss=0.0587 | val_loss=0.2212 | val_auc=0.9783 | val_f1=0.9173
Fold 01 | Epoch 004/30 | train_loss=0.0355 | val_loss=0.3134 | val_auc=0.9681 | val_f1=0.9124
Fold 01 | Epoch 005/30 | train_loss=0.0302 | val_loss=0.2531 | val_auc=0.9784 | val_f1=0.9175
Fold 01 | Epoch 006/30 | train_loss=0.0298 | val_loss=0.3030 | val_auc=0.9766 | val_f1=0.9206
Fold 01 | Epoch 007/30 | train_loss=0.0207 | val_loss=0.3577 | val_auc=0.9695 | val_f1=0.9131
Fold 01 | Epoch 008/30 | train_loss=0.0130 | val_loss=0.3404 | val_auc=0.9728 | val_f1=0.8964
Fold 01 | Epoch 009/30 | train_loss=0.0092 | val_loss=0.4163 | val_auc=0.9703 | val_f1=0.8934
Fold 01 | Epoch 010/30 | train_loss=0.0064 | val_loss=0.3880 | val_auc=0.9750 | val_f1=0.8960
Early stopping no fold 1: 8 épocas sem melhora.
[Fold 1] PCA